# 01: Baselines, splits, data, label noise

Tasks 1, 2, 3 and 5 of the task description. The setup cells below are given: read them, then
run them. **The experiments are yours** (a coding agent may help).

Every experiment has the same three steps:
1. ✍️ write down your prediction,
2. run it,
3. ✍️ explain the result, especially where your prediction was wrong.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from load import load, features, GESTURES, FEATURE_NAMES   # load.py next to this notebook

# Download the class data once, so your numbers stay comparable while others keep recording:
#   curl -o gestures.zip "<export link from the projector>"
SRC = "gestures.zip"
X, y, person = load(SRC)
people = np.unique(person)
print(X.shape, "= takes x time steps (50 Hz) x [acc_x acc_y acc_z gyro_x gyro_y gyro_z]")

## Given helpers
`evaluate(...)`, `nn_same_person(...)`, `first_reps(k)`, `some_people(k)`, `flip_labels(frac)`. Read the docstrings.

In [ ]:
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.neighbors import NearestNeighbors

F = features(X)          # (N, 30): mean, std, min, max, energy per channel; names in FEATURE_NAMES


def splits(kind, n_splits=5, seed=0):
    """Yield (train_idx, test_idx) for 5 folds.
    kind="random": takes shuffled at random.  kind="person": nobody is in both train and test."""
    idx = np.arange(len(y))
    if kind == "random":
        yield from StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(idx, y)
    elif kind == "person":
        yield from GroupKFold(min(n_splits, len(people))).split(idx, y, person)
    else:
        raise ValueError(kind)


def evaluate(make_model, Z, kind, train_filter=None, y_train=None):
    """Mean test accuracy over the folds.
    make_model:   function returning a fresh model, e.g. lambda: RandomForestClassifier(300)
    Z:            inputs, e.g. F (features) or X.reshape(len(X), -1) (raw window)
    train_filter: optional, shrinks the training indices of each fold (task 3)
    y_train:      optional, labels to train on instead of y (task 5); tests always use the true y"""
    y_fit = y if y_train is None else y_train
    accs = []
    for tr, te in splits(kind):
        if train_filter is not None:
            tr = train_filter(tr)
        model = make_model().fit(Z[tr], y_fit[tr])
        accs.append((model.predict(Z[te]) == y[te]).mean())
    return float(np.mean(accs))


def nn_same_person(Z, kind):
    """How often is the nearest training take (Euclidean on Z) from the same person as the test take?"""
    same = []
    for tr, te in splits(kind):
        nn = NearestNeighbors(n_neighbors=1).fit(Z[tr]).kneighbors(Z[te], return_distance=False)[:, 0]
        same.append((person[tr][nn] == person[te]).mean())
    return float(np.mean(same))


# task 3: which repetition of (person, gesture) each take is: 0, 1, 2, ...
rep = np.zeros(len(y), int)
for p in people:
    for g in GESTURES:
        m = np.where((person == p) & (y == g))[0]
        rep[m] = np.arange(len(m))


def first_reps(k):
    """train_filter: keep only the first k reps of every person and gesture."""
    return lambda tr: tr[rep[tr] < k]


def some_people(k, seed=0):
    """train_filter: keep only k random people of the training fold."""
    def f(tr):
        ps = np.unique(person[tr])
        keep = np.random.default_rng(seed).choice(ps, min(k, len(ps)), replace=False)
        return tr[np.isin(person[tr], keep)]
    return f


def flip_labels(frac, seed=0):
    """Copy of y with a fraction of the labels changed to a random *other* gesture (task 5)."""
    rng = np.random.default_rng(seed)
    y_noisy = y.copy()
    for i in rng.choice(len(y), int(frac * len(y)), replace=False):
        y_noisy[i] = rng.choice([g for g in GESTURES if g != y[i]])
    return y_noisy

print(f"{len(y)} takes, {len(people)} people, {F.shape[1]} features")

## Task 1: Baseline
Train a logistic regression and a random forest on `F`, and kNN on the standardized features (`make_pipeline(StandardScaler(), KNeighborsClassifier(5))`). Use `evaluate(..., kind="random")` for now.

**✍️ Your prediction (before running anything):** Which of the three is best, and roughly what accuracy? (6 classes: chance is 0.17)

> 

In [ ]:
# your code


Which gestures get confused? (`sklearn.metrics.ConfusionMatrixDisplay`)

In [ ]:
# your code


**✍️ Result and explanation:** What happened, and why?

> 

## Task 2: Split matters
Same models, `kind="person"`: the test people are never seen in training.

**✍️ Your prediction (before running anything):** Accuracy with the per-person split, for each model? Bigger or smaller gap for kNN?

> 

In [ ]:
# your code


For each test take, whose take is its nearest neighbor? Use `nn_same_person(Z, kind)` with standardized features.

**✍️ Your prediction (before running anything):** How often is the nearest neighbor the same person, under each split?

> 

In [ ]:
# your code


**✍️ Result and explanation:** Why the gap between the splits? Which accuracy would you report, and why?

> 

## Task 3: How much data
Keep `kind="person"`. Shrink only the training data:
- reps: `train_filter=first_reps(k)` for k = 1, 2, 5, 10, 15
- people: `train_filter=some_people(k)` for k = 1, 2, 4, 8, ...

Plot accuracy against the number of training takes, both curves on the same axes.

**✍️ Your prediction (before running anything):** Which helps more: more reps from the same people, or more people?

> 

In [ ]:
# your code


**✍️ Result and explanation:** What happened, and why?

> 

## Task 5: Label noise
Train on `y_train=flip_labels(frac)` for frac = 0, 0.1, 0.2, 0.4. The test labels stay clean.

**✍️ Your prediction (before running anything):** Accuracy at 40% wrong labels? Does the forest or kNN suffer more?

> 

In [ ]:
# your code


**✍️ Result and explanation:** What happened, and why?

> 

## Report
Add your pair's row to the class results table: model, split, accuracy with `kind="person"`, number of training takes.